# Homemade RecipeBowl - Text to Recipe Generator

This notebook trains a Seq2Seq Transformer model that takes ingredients/prompts as input and generates a complete recipe structure.
Instructions for Kaggle:
1. Turn on GPU T4x2 (Settings -> Accelerator -> GPU T4x2).
2. Add a Recipe dataset, e.g., 'Recipe1M+' or 'Epicurious'.
3. Run all cells.
4. Download the `text_model.pt` at the end and place it in the `backend/models/` folder of the Flask app.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
from transformers import GPT2Tokenizer, GPT2LMHeadModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 1. Load Dataset (Example with generic CSV structure)
# Please replace the path with your Kaggle dataset path
try:
    df = pd.read_csv('/kaggle/input/recipe-dataset/recipes.csv')
    print(f"Loaded {len(df)} recipes.")
except FileNotFoundError:
    print("Dataset not found. Please attach a dataset in Kaggle.")
    # Dummy dataframe for testing the code
    df = pd.DataFrame({
        'ingredients': ['eggs, milk', 'chicken, salt, pepper'],
        'instructions': ['Scramble eggs with milk.', 'Bake chicken at 350F.']
    })

In [ ]:
# 2. Tokenizer Setup (Using pre-trained GPT2 Tokenizer for ease of generation)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# 3. Dataset Setup
class RecipeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.max_length = max_length
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        row = self.data.iloc[index]
        # Format: <INGREDIENTS> item1, item2 <RECIPE> step 1... <EOS>
        text = f"<INGREDIENTS> {row['ingredients']} <RECIPE> {row['instructions']} <EOS>"
        
        encodings = self.tokenizer(text, truncation=True, max_length=self.max_length, padding="max_length", return_tensors='pt')
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze()
        }

dataset = RecipeDataset(df, tokenizer)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

In [ ]:
# 4. Model Setup
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
epochs = 3

In [ ]:
# 5. Training Loop
print("Starting training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        labels = input_ids.clone()
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Average Loss: {total_loss / len(dataloader):.4f}")

In [ ]:
# 6. Export Model for Backend Inference
torch.save(model.state_dict(), 'text_model.pt')
print("Model saved as text_model.pt. Download this file and copy it to backend/models/")